In [6]:

!pip -q install datasets pandas scikit-learn


In [9]:

import pandas as pd
import requests

languages = {
    "yor": "Yoruba",
    "hau": "Hausa",
    "ibo": "Igbo"
}

records = []

base_url = (
    "https://raw.githubusercontent.com/"
    "masakhane-io/masakhane-ner/main/"
    "MasakhaNER2.0/data"
)

for code, language in languages.items():
    url = f"{base_url}/{code}/train.txt"

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    sentences = []
    words = []

    for line in response.text.splitlines():
        line = line.strip()

        if not line:
            if words:
                sentences.append(" ".join(words))
                words = []
        else:
            if not line.startswith("#"):
                words.append(line.split()[0])

    if words:
        sentences.append(" ".join(words))

    for sentence in sentences:
        if len(sentence.split()) >= 3:
            records.append({
                "text": sentence,
                "language": language,
                "source": "MasakhaNER2"
            })

df = pd.DataFrame(records)

df = df.drop_duplicates(subset=["text"])

print("DATASET DOWNLOADED SUCCESSFULLY!")
print(df["language"].value_counts())

display(df.head(10))


DATASET DOWNLOADED SUCCESSFULLY!
language
Igbo      7493
Yoruba    6643
Hausa     5680
Name: count, dtype: int64


,text,language,source
0,Bàbá tó bí Bàbá Gàní Adams ilẹ̀ Yorùb...,Yoruba,MasakhaNER2
1,Amọ ìgbésẹ̀ tó ń gbé ó fẹ́ẹ́ ma sọ pé e...,Yoruba,MasakhaNER2
2,"Akínkanjú nìran ààrẹ ọ̀nà kakaǹfò , ko...",Yoruba,MasakhaNER2
3,Kí ló wá dé tóun joyè ààrẹ tó ń sá ...,Yoruba,MasakhaNER2
4,"kò bá má ti gboyè yẹn , kí wọ́n gbe fẹ́ni...",Yoruba,MasakhaNER2
5,Ààrẹ ònà kakaǹfò kìí sọ̀rọ̀ ká kàn j...,Yoruba,MasakhaNER2
6,Ohun tí Yorùbá sọ ni pé ẹ sinmi ariwo tí ...,Yoruba,MasakhaNER2
7,"Lọ́jọ́ orí ọ̀pọ̀ wa ló jù ẹ́ lọ , ṣùgbọ́n ...",Yoruba,MasakhaNER2
8,"Ẹni bá jẹ́ ọmọ àlè nínú wa , á bógun lọ...",Yoruba,MasakhaNER2
9,Èèyàn kìí wà nípò òótọ́ kó kú síp...,Yoruba,MasakhaNER2


In [10]:

df.to_csv(
    "nigerian_languages_initial.csv",
    index=False,
    encoding="utf-8-sig"
)

from google.colab import files

files.download("nigerian_languages_initial.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:

print("LANGUAGE DATASET SUMMARY")
print("------------------------")

print(df["language"].value_counts())

print("\nTotal samples:", len(df))


LANGUAGE DATASET SUMMARY
------------------------
language
Igbo      7493
Yoruba    6643
Hausa     5680
Name: count, dtype: int64

Total samples: 19816


In [12]:

import nltk
import pandas as pd

nltk.download("brown", quiet=True)

from nltk.corpus import brown

# Get English sentences
english_sentences = [
    " ".join(sentence)
    for sentence in brown.sents()
    if 3 <= len(sentence) <= 30
]

# Remove duplicate sentences
english_sentences = list(dict.fromkeys(english_sentences))

# Keep 6,000 English samples
english_sentences = english_sentences[:6000]

english_df = pd.DataFrame({
    "text": english_sentences,
    "language": "English",
    "source": "NLTK Brown Corpus"
})

# Add English to existing dataset
df = pd.concat(
    [df, english_df],
    ignore_index=True
)

print("ENGLISH DATA ADDED SUCCESSFULLY!")
print(df["language"].value_counts())


ENGLISH DATA ADDED SUCCESSFULLY!
language
Igbo       7493
Yoruba     6643
English    6000
Hausa      5680
Name: count, dtype: int64


In [13]:

# Remove empty and duplicate text
df = df.dropna(subset=["text", "language"]).copy()
df["text"] = df["text"].astype(str).str.strip()

df = df[df["text"].str.len() > 0]

df = df.drop_duplicates(
    subset=["text"]
)

# Select 5,000 samples per language
balanced_df = (
    df.groupby("language", group_keys=False)
    .sample(n=5000, random_state=42)
    .reset_index(drop=True)
)

# Shuffle the dataset
balanced_df = balanced_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("FINAL DATASET SUMMARY")
print(balanced_df["language"].value_counts())
print("Total samples:", len(balanced_df))

display(balanced_df.head(10))


FINAL DATASET SUMMARY
language
Igbo       5000
English    5000
Hausa      5000
Yoruba     5000
Name: count, dtype: int64
Total samples: 20000


,text,language,source
0,O kwùkwazịrị na nnyocha ka na - aga n'ihu bany...,Igbo,MasakhaNER2
1,A normal year's work in college is 30 semester...,English,NLTK Brown Corpus
2,Babu wani dalibina da ya rasu a hannun yan bin...,Hausa,MasakhaNER2
3,In addition to urging greater restrictions on ...,English,NLTK Brown Corpus
4,Nke a bụ Oziọma Ọdịnanị m ji were bịa ta .,Igbo,MasakhaNER2
5,"Maazị Zakariah Idzy , onye steeti Kaduna bụ ot...",Igbo,MasakhaNER2
6,Mmemme ahụ bụ nke were otu izuụka .,Igbo,MasakhaNER2
7,They did not view the tour of the distressed c...,English,NLTK Brown Corpus
8,Ó ro̩ àwo̩n o̩mo̩ Nàìjíríà láti má bo...,Yoruba,MasakhaNER2
9,"Abduslami Abubakar , wanda yake shugaban kwami...",Hausa,MasakhaNER2


In [15]:

from sklearn.model_selection import train_test_split

X = balanced_df["text"]
y = balanced_df["language"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 16000
Testing samples: 4000


In [14]:

from google.colab import files

balanced_df.to_csv(
    "Nigerian_Language_Identifier_Dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

files.download(
    "Nigerian_Language_Identifier_Dataset.csv"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 5),
        min_df=2,
        max_features=100000
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

print("Training model...")

model.fit(X_train, y_train)

print("MODEL TRAINING COMPLETED!")


Training model...
MODEL TRAINING COMPLETED!


In [17]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("MODEL ACCURACY REPORT")
print("---------------------")
print(f"Accuracy: {accuracy * 100:.2f}%")

print("\nDetailed Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))


MODEL ACCURACY REPORT
---------------------
Accuracy: 99.70%

Detailed Classification Report:
              precision    recall  f1-score   support

     English       0.99      1.00      1.00      1000
       Hausa       1.00      1.00      1.00      1000
        Igbo       1.00      1.00      1.00      1000
      Yoruba       1.00      0.99      1.00      1000

    accuracy                           1.00      4000
   macro avg       1.00      1.00      1.00      4000
weighted avg       1.00      1.00      1.00      4000



In [19]:

test_phrases = [
    "How are you today?",
    "Ẹ káàárọ̀",
    "Ina kwana",
    "Kedu ka ị mere",
    "I love learning new things"
]

print("LANGUAGE PREDICTION TEST")
print("------------------------")

for phrase in test_phrases:
    prediction = model.predict([phrase])[0]
    print(f"{phrase} --> {prediction}")


LANGUAGE PREDICTION TEST
------------------------
How are you today? --> English
Ẹ káàárọ̀ --> Yoruba
Ina kwana --> Hausa
Kedu ka ị mere --> Igbo
I love learning new things --> English


In [20]:

import joblib
from google.colab import files

joblib.dump(
    model,
    "nigerian_language_identifier_model.pkl"
)

print("Model saved successfully!")

files.download(
    "nigerian_language_identifier_model.pkl"
)


Model saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:

!pip -q install gradio


In [22]:

import gradio as gr

def identify_language(text):
    if not text or not text.strip():
        return "Please enter some text."

    prediction = model.predict([text])[0]

    return f"Predicted Language: {prediction}"

app = gr.Interface(
    fn=identify_language,
    inputs=gr.Textbox(
        lines=3,
        label="Enter your text",
        placeholder="Type in Yoruba, Hausa, Igbo or English..."
    ),
    outputs=gr.Textbox(
        label="Language Prediction"
    ),
    title="Nigerian Language Identifier",
    description=(
        "A machine learning application that identifies "
        "Yoruba, Hausa, Igbo and English text."
    ),
    examples=[
        ["How are you today?"],
        ["Ẹ káàárọ̀"],
        ["Ina kwana"],
        ["Kedu ka ị mere"]
    ]
)

app.launch()



It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://afac5fca09877e0fa4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:

from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("OVERALL ACCURACY:")
print(f"{accuracy_score(y_test, y_pred) * 100:.2f}%")

print("\nCLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred))


OVERALL ACCURACY:
99.70%

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

     English       0.99      1.00      1.00      1000
       Hausa       1.00      1.00      1.00      1000
        Igbo       1.00      1.00      1.00      1000
      Yoruba       1.00      0.99      1.00      1000

    accuracy                           1.00      4000
   macro avg       1.00      1.00      1.00      4000
weighted avg       1.00      1.00      1.00      4000



In [24]:

test_data = [
    ("Good morning", "English"),
    ("Where are you going?", "English"),
    ("Thank you very much", "English"),
    ("I am coming home", "English"),

    ("Ẹ kú iṣẹ́", "Yoruba"),
    ("Mo fẹ́ jẹun", "Yoruba"),
    ("Báwo ni ilé?", "Yoruba"),
    ("Mo dúpẹ́", "Yoruba"),

    ("Sannu da zuwa", "Hausa"),
    ("Na gode sosai", "Hausa"),
    ("Ina son ruwa", "Hausa"),
    ("Yaya iyali?", "Hausa"),

    ("Ụtụtụ ọma", "Igbo"),
    ("Daalụ nke ukwuu", "Igbo"),
    ("Achọrọ m mmiri", "Igbo"),
    ("Ebee ka ị na-aga?", "Igbo")
]

correct = 0

print("REAL-WORLD PHRASE TEST")
print("-" * 55)

for phrase, actual in test_data:
    predicted = model.predict([phrase])[0]

    status = "CORRECT" if predicted == actual else "WRONG"

    if predicted == actual:
        correct += 1

    print(
        f"{phrase} | Actual: {actual} | "
        f"Predicted: {predicted} | {status}"
    )

print("\nFINAL RESULTS")
print("Correct predictions:", correct)
print("Total phrases:", len(test_data))
print(f"Accuracy: {correct / len(test_data) * 100:.2f}%")


REAL-WORLD PHRASE TEST
-------------------------------------------------------
Good morning | Actual: English | Predicted: English | CORRECT
Where are you going? | Actual: English | Predicted: English | CORRECT
Thank you very much | Actual: English | Predicted: English | CORRECT
I am coming home | Actual: English | Predicted: English | CORRECT
Ẹ kú iṣẹ́ | Actual: Yoruba | Predicted: Yoruba | CORRECT
Mo fẹ́ jẹun | Actual: Yoruba | Predicted: Yoruba | CORRECT
Báwo ni ilé? | Actual: Yoruba | Predicted: Yoruba | CORRECT
Mo dúpẹ́ | Actual: Yoruba | Predicted: Yoruba | CORRECT
Sannu da zuwa | Actual: Hausa | Predicted: Hausa | CORRECT
Na gode sosai | Actual: Hausa | Predicted: English | WRONG
Ina son ruwa | Actual: Hausa | Predicted: English | WRONG
Yaya iyali? | Actual: Hausa | Predicted: Hausa | CORRECT
Ụtụtụ ọma | Actual: Igbo | Predicted: Igbo | CORRECT
Daalụ nke ukwuu | Actual: Igbo | Predicted: Igbo | CORRECT
Achọrọ m mmiri | Actual: Igbo | Predicted: Igbo | CORRECT
Ebee ka ị na-aga? |

In [25]:

import pandas as pd

phrases = [
    "Na gode sosai",
    "Ina son ruwa",
    "Sannu da zuwa",
    "Yaya iyali?"
]

probabilities = model.predict_proba(phrases)
languages = model.classes_

results = pd.DataFrame(
    probabilities,
    columns=languages,
    index=phrases
)

print("LANGUAGE PREDICTION PROBABILITIES")
display((results * 100).round(2))


LANGUAGE PREDICTION PROBABILITIES


,English,Hausa,Igbo,Yoruba
Na gode sosai,41.87,19.41,20.45,18.27
Ina son ruwa,35.69,31.73,14.32,18.26
Sannu da zuwa,12.58,68.42,7.90,11.10
Yaya iyali?,20.77,47.92,15.14,16.18


In [26]:

phrases_to_check = [
    "Na gode sosai",
    "Ina son ruwa",
    "Sannu da zuwa",
    "Yaya iyali?"
]

for phrase in phrases_to_check:
    matches = balanced_df[
        balanced_df["text"].str.contains(
            phrase,
            case=False,
            regex=False,
            na=False
        )
    ]

    print(f"\nPhrase: {phrase}")
    print(f"Matching dataset rows: {len(matches)}")

    if not matches.empty:
        display(matches.head())



Phrase: Na gode sosai
Matching dataset rows: 0

Phrase: Ina son ruwa
Matching dataset rows: 0

Phrase: Sannu da zuwa
Matching dataset rows: 0

Phrase: Yaya iyali?
Matching dataset rows: 0


In [27]:

import pandas as pd

phrases = [
    "Na gode sosai",
    "Ina son ruwa",
    "Sannu da zuwa",
    "Yaya iyali?"
]

probabilities = model.predict_proba(phrases)

results = pd.DataFrame(
    probabilities * 100,
    columns=model.classes_,
    index=phrases
)

print("PREDICTION PROBABILITIES (%)")
display(results.round(2))


PREDICTION PROBABILITIES (%)


,English,Hausa,Igbo,Yoruba
Na gode sosai,41.87,19.41,20.45,18.27
Ina son ruwa,35.69,31.73,14.32,18.26
Sannu da zuwa,12.58,68.42,7.90,11.10
Yaya iyali?,20.77,47.92,15.14,16.18


In [28]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# Create an alternative model
svm_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 5),
        min_df=2,
        max_features=100000
    )),
    ("classifier", LinearSVC(
        random_state=42
    ))
])

print("Training Linear SVM model...")

svm_model.fit(X_train, y_train)

print("SVM MODEL TRAINING COMPLETED!")

# Evaluate on the original test set
svm_predictions = svm_model.predict(X_test)

print("\nSVM TEST ACCURACY:")
print(
    f"{accuracy_score(y_test, svm_predictions) * 100:.2f}%"
)

print("\nCLASSIFICATION REPORT:")
print(classification_report(
    y_test,
    svm_predictions,
    zero_division=0
))


Training Linear SVM model...
SVM MODEL TRAINING COMPLETED!

SVM TEST ACCURACY:
99.80%

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

     English       0.99      1.00      1.00      1000
       Hausa       1.00      1.00      1.00      1000
        Igbo       1.00      1.00      1.00      1000
      Yoruba       1.00      1.00      1.00      1000

    accuracy                           1.00      4000
   macro avg       1.00      1.00      1.00      4000
weighted avg       1.00      1.00      1.00      4000



In [29]:

print("COMPARING BOTH MODELS")
print("-" * 70)

lr_correct = 0
svm_correct = 0

for phrase, actual in test_data:
    lr_prediction = model.predict([phrase])[0]
    svm_prediction = svm_model.predict([phrase])[0]

    if lr_prediction == actual:
        lr_correct += 1

    if svm_prediction == actual:
        svm_correct += 1

    print(f"Phrase: {phrase}")
    print(f"Actual: {actual}")
    print(f"Logistic Regression: {lr_prediction}")
    print(f"Linear SVM: {svm_prediction}")
    print("-" * 70)

print("\nFINAL COMPARISON")
print(
    f"Logistic Regression: "
    f"{lr_correct}/{len(test_data)} correct"
)
print(
    f"Linear SVM: "
    f"{svm_correct}/{len(test_data)} correct"
)


COMPARING BOTH MODELS
----------------------------------------------------------------------
Phrase: Good morning
Actual: English
Logistic Regression: English
Linear SVM: English
----------------------------------------------------------------------
Phrase: Where are you going?
Actual: English
Logistic Regression: English
Linear SVM: English
----------------------------------------------------------------------
Phrase: Thank you very much
Actual: English
Logistic Regression: English
Linear SVM: English
----------------------------------------------------------------------
Phrase: I am coming home
Actual: English
Logistic Regression: English
Linear SVM: English
----------------------------------------------------------------------
Phrase: Ẹ kú iṣẹ́
Actual: Yoruba
Logistic Regression: Yoruba
Linear SVM: Yoruba
----------------------------------------------------------------------
Phrase: Mo fẹ́ jẹun
Actual: Yoruba
Logistic Regression: Yoruba
Linear SVM: Yoruba
---------------------------

In [30]:

import joblib
from google.colab import files

# Save the selected Linear SVM model
joblib.dump(
    svm_model,
    "nigerian_language_identifier_svm.pkl"
)

print("LINEAR SVM MODEL SAVED SUCCESSFULLY!")

# Download the model
files.download(
    "nigerian_language_identifier_svm.pkl"
)


LINEAR SVM MODEL SAVED SUCCESSFULLY!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:

import gradio as gr

def identify_language(text):
    if not text or not text.strip():
        return "Please enter some text."

    prediction = svm_model.predict([text])[0]

    return f"Predicted Language: {prediction}"

app = gr.Interface(
    fn=identify_language,
    inputs=gr.Textbox(
        lines=3,
        label="Enter Your Text",
        placeholder="Type a phrase in Yoruba, Hausa, Igbo or English..."
    ),
    outputs=gr.Textbox(
        label="Predicted Language"
    ),
    title="Nigerian Language Identifier",
    description=(
        "A machine learning application that identifies "
        "Yoruba, Hausa, Igbo and English text using "
        "a trained Linear Support Vector Machine."
    ),
    examples=[
        ["How are you today?"],
        ["Ẹ káàárọ̀"],
        ["Sannu da zuwa"],
        ["Kedu ka ị mere"]
    ]
)

app.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a48a6ada21ad1e3f1c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
